### Review the structured data

In [22]:
import sqlite3
import pandas as pd
import os
import re
import json
import warnings
from dotenv import load_dotenv
from sqlalchemy import text
from rank_bm25 import BM25Okapi
from pathlib import Path
from typing import List, Dict, Any

# LangChain Imports
from langchain_openai import OpenAI
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [23]:
DB_PATH = Path(
    r"E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\structured_data\structured.sqlite"
)

assert DB_PATH.exists(), "❌ SQLite file not found"
print(f"✅ Using DB: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)

✅ Using DB: E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\structured_data\structured.sqlite


In [24]:
tables_df = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)

tables_df


,name
0,advanced_ratios
1,expense_exit_tax
2,fund_managers
3,holdings
4,minimum_investment
5,nav_summary
6,return_periods
7,return_values
8,returns_meta
9,schemes


In [25]:
def preview_table(table_name, conn, limit=10):
    print("=" * 80)
    print(f"📋 TABLE: {table_name}")
    print("=" * 80)

    # Row count
    count_df = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table_name}", conn)
    print(f"Total rows: {count_df.loc[0, 'row_count']}")

    # Preview data
    df = pd.read_sql(f"SELECT * FROM {table_name} LIMIT {limit}", conn)

    print("\nColumns:")
    print(list(df.columns))

    print("\nSample rows:")
    display(df)

In [26]:
for table_name in tables_df["name"]:
    preview_table(table_name, conn)

📋 TABLE: advanced_ratios
Total rows: 6

Columns:
['scheme_id', 'sharpe_ratio', 'sortino_ratio', 'alpha', 'beta', 'standard_deviation', 'information_ratio', 'risk_rating', 'risk']

Sample rows:


,scheme_id,sharpe_ratio,sortino_ratio,alpha,beta,standard_deviation,information_ratio,risk_rating,risk
0,parag-parikh-long-term-value-fund-direct-growth,1.875281,2.787378,9.587596,0.554315,8.038466,0.757558,6,Very High
1,parag-parikh-liquid-fund-direct-growth,2.102005,3.500306,0.861854,0.324757,0.150153,9.069900,7,Low to Moderate
2,parag-parikh-elss-tax-saver-fund-direct-growth,1.252944,1.979923,4.909543,0.649564,9.049992,0.252651,4,Moderately High
3,parag-parikh-conservative-hybrid-fund-direct-g...,1.719147,2.558328,3.296704,0.634979,3.274656,0.768176,4,Moderately High
4,parag-parikh-arbitrage-fund-direct-growth,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,Low
5,parag-parikh-dynamic-asset-allocation-fund-dir...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3,Moderate


📋 TABLE: expense_exit_tax
Total rows: 6

Columns:
['scheme_id', 'expense_ratio', 'exit_load', 'stamp_duty', 'tax_implication']

Sample rows:


,scheme_id,expense_ratio,exit_load,stamp_duty,tax_implication
0,parag-parikh-long-term-value-fund-direct-growth,0.63%,"For units above 10% of the investment, exit lo...","0.005% (from July 1st, 2020)","Returns are taxed at 20%, if you redeem before..."
1,parag-parikh-liquid-fund-direct-growth,0.11%,"Exit load of 0.0070% if redeemed within 1 day,...","0.005% (from July 1st, 2020)",Returns are taxed as per your Income Tax slab....
2,parag-parikh-elss-tax-saver-fund-direct-growth,0.62%,Nil,"0.005% (from July 1st, 2020)","Returns are taxed at 20%, if you redeem before..."
3,parag-parikh-conservative-hybrid-fund-direct-g...,0.34%,"For units in excess of 10% of the investment,1...","0.005% (from July 1st, 2020)",Returns are taxed as per your Income Tax slab....
4,parag-parikh-arbitrage-fund-direct-growth,0.30%,"Exit load of 0.25%, if redeemed within 30 days.","0.005% (from July 1st, 2020)","Returns are taxed at 20%, if you redeem before..."
5,parag-parikh-dynamic-asset-allocation-fund-dir...,0.33%,Exit Load for units in excess of 10% of the in...,"0.005% (from July 1st, 2020)",NA Understand terms Check past data


📋 TABLE: fund_managers
Total rows: 34

Columns:
['id', 'scheme_id', 'manager_name', 'tenure', 'education', 'experience']

Sample rows:


...

,scheme_id,category,active_type
0,parag-parikh-long-term-value-fund-direct-growth,Equity Flexi Cap,Annualised returns
1,parag-parikh-liquid-fund-direct-growth,Debt Liquid,Annualised returns
2,parag-parikh-elss-tax-saver-fund-direct-growth,Equity ELSS,Annualised returns
3,parag-parikh-conservative-hybrid-fund-direct-g...,Hybrid Conservative Hybrid,Annualised returns
4,parag-parikh-arbitrage-fund-direct-growth,Hybrid Arbitrage,Absolute returns
5,parag-parikh-dynamic-asset-allocation-fund-dir...,Hybrid Dynamic Asset Allocation,Absolute returns


📋 TABLE: schemes
Total rows: 6

Columns:
['scheme_id', 'scheme_name', 'category', 'sub_category', 'risk', 'amc_name']

Sample rows:


,scheme_id,scheme_name,category,sub_category,risk,amc_name
0,parag-parikh-long-term-value-fund-direct-growth,Parag Parikh Flexi Cap Fund Direct Growth,Equity,Flexi Cap,Very High Risk,PPFAS Mutual Fund
1,parag-parikh-liquid-fund-direct-growth,Parag Parikh Liquid Fund Direct Growth,Debt,Liquid,Low to Moderate Risk,PPFAS Mutual Fund
2,parag-parikh-elss-tax-saver-fund-direct-growth,Parag Parikh ELSS Tax Saver Fund Direct Growth,Equity,ELSS,Moderately High Risk,PPFAS Mutual Fund
3,parag-parikh-conservative-hybrid-fund-direct-g...,Parag Parikh Conservative Hybrid Fund Direct G...,Hybrid,Conservative Hybrid,Moderately High Risk,PPFAS Mutual Fund
4,parag-parikh-arbitrage-fund-direct-growth,Parag Parikh Arbitrage Fund Direct Growth,Hybrid,Arbitrage,Low Risk,PPFAS Mutual Fund
5,parag-parikh-dynamic-asset-allocation-fund-dir...,Parag Parikh Dynamic Asset Allocation Fund Dir...,Hybrid,Dynamic Asset Allocation,Moderate Risk,PPFAS Mutual Fund


📋 TABLE: sqlite_sequence
Total rows: 4

Columns:
['name', 'seq']

Sample rows:


,name,seq
0,return_periods,24
1,return_values,72
2,holdings,869
3,fund_managers,34


### Structured RAG

In [27]:
# ----------------------------
# ENV SETUP
# ----------------------------
load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
SQLITE_DB_PATH = os.environ.get("SQLITE_DB_PATH")

In [28]:
# ----------------------------
# GLOBAL VARIABLES
# ----------------------------
AMC_NAME = "PPFAS Mutual Fund"
SOURCE_LINK = "https://groww.in/mutual-funds/amc/ppfas-mutual-funds"
JSON_PATH = Path(r"E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files\chunkable_ppfas.json")

# Dynamic Date Fetching
try:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        _data = json.load(f)
        LAST_UPDATED_DATE = _data[0].get("overview", {}).get("nav", {}).get("as_on", "Unknown Date")
        print(f"✅ LAST_UPDATED_DATE = {LAST_UPDATED_DATE}")
except Exception as e:
    print(f"⚠️ Could not load LAST_UPDATED_DATE: {e}")
    LAST_UPDATED_DATE = "Unknown Date"

✅ LAST_UPDATED_DATE = 02 Jan 2026


In [29]:
# ----------------------------
# DATABASE
# ----------------------------
# Ensure this run is only if paths are set
if SQLITE_DB_PATH:
    db = SQLDatabase.from_uri(f"sqlite:///{SQLITE_DB_PATH}")
else:
    db = None
    print("⚠️ SQLITE_DB_PATH not set. Database features will fail.")

In [30]:
# ----------------------------
# LLM (INTENT ONLY)
# ----------------------------
llm = OpenAI(
    model_name="gpt-4o-mini",
    temperature=0
)

In [31]:
# ----------------------------
# PII MIDDLEWARE
# ----------------------------
PII_PATTERNS = [
    r"\b[A-Z]{5}[0-9]{4}[A-Z]\b",                  # PAN
    r"\b[2-9]{1}[0-9]{3}\s[0-9]{4}\s[0-9]{4}\b",  # Aadhaar (fmt 1)
    r"\b[2-9]{1}[0-9]{3}[0-9]{4}[0-9]{4}\b",      # Aadhaar (fmt 2)
    r"\b\d{10}\b",                                # Phone
    r"\b(\+91[\-\s]?)?[0]?(91)?[789]\d{9}\b",     # Phone (India)
    r"\b[\w\.-]+@[\w\.-]+\.\w+\b",                # Email
    r"\b\d{9,18}\b",                              # Account numbers
    r"\b\d{4,6}\b"                                # OTP/PIN
]

PII_REFUSAL_MESSAGE = (
    "For your privacy and security, I can’t process questions that include sensitive personal information.\n"
    "Please remove any personal identifiers and ask your question again in a general form.\n\n"
    "I answer only factual mutual fund queries related to Parag Parikh Mutual Fund schemes using official Groww information:\n"
    f"{SOURCE_LINK}"
)

def detect_pii(text: str) -> bool:
    return any(re.search(p, text, re.IGNORECASE) for p in PII_PATTERNS)

In [32]:
# ----------------------------
# SCHEME LOADING & BM25
# ----------------------------
def load_schemes(database: SQLDatabase):
    if not database: return []
    engine = database._engine
    with engine.connect() as conn:
        rows = conn.execute(
            text("SELECT scheme_id, scheme_name FROM schemes")
        ).fetchall()
    return [{"scheme_id": r[0], "scheme_name": r[1]} for r in rows]

SCHEMES = load_schemes(db)

ALIAS_MAP = {
    "ppfas": "parag parikh",
    "elss": "tax saver",
    "taxsaver": "tax saver",
    "long term": "flexi cap",
}

def canonicalize(text: str) -> str:
    t = text.lower()
    for k, v in ALIAS_MAP.items():
        t = t.replace(k, v)
    return t

if SCHEMES:
    bm25_corpus = [s["scheme_name"].lower().split() for s in SCHEMES]
    bm25 = BM25Okapi(bm25_corpus)
else:
    bm25 = None

In [33]:
# ----------------------------
# INTENTS
# ----------------------------
SINGLE_SCHEME_INTENTS = {
    "NAV", "MIN_SIP", "EXPENSE_RATIO", "EXIT_LOAD", "FUND_MANAGER",
    "RETURNS", "HOLDINGS", "RISK_RATIOS"
}

STRUCTURED_INTENTS = {
    "NAV", "MIN_SIP", "EXPENSE_RATIO", "EXIT_LOAD", "FUND_MANAGER",
    "RETURNS", "HOLDINGS", "RISK_RATIOS"
}

def resolve_schemes(query: str, intent: str, top_k: int = 3):
    if not bm25: return []
    q = canonicalize(query)
    query_tokens = q.split()
    scores = bm25.get_scores(query_tokens)
    
    ranked = sorted(zip(SCHEMES, scores), key=lambda x: x[1], reverse=True)
    
    if "all" in q or "schemes" in q:
        return SCHEMES
        
    if intent in SINGLE_SCHEME_INTENTS:
        best_scheme, best_score = ranked[0]
        if best_score == 0: return []
        return [best_scheme]
        
    return [scheme for scheme, score in ranked[:top_k] if score > 0]

def get_text_from_response(response) -> str:
    if isinstance(response, str): return response
    return response.content

INTENT_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""
Classify the user's question into ONE intent.

STRUCTURED (factual, numeric, database):
- NAV
- MIN_SIP
- EXPENSE_RATIO
- EXIT_LOAD
- FUND_MANAGER
- RETURNS
- HOLDINGS
- RISK_RATIOS

UNSTRUCTURED (explanatory, descriptive):
- INVESTMENT_PHILOSOPHY
- INVESTMENT_APPROACH
- GENERAL_INFO

OTHER:
- OPINIONATED
- VAGUE
- UNKNOWN

User question:
{question}

Return ONLY the intent label.
"""
)

def detect_intent(question: str) -> str:
    response = llm.invoke(INTENT_PROMPT.format(question=question))
    return get_text_from_response(response).strip().upper()

In [34]:
# ----------------------------
# SQL HANDLERS
# ----------------------------
def run_query(query: str, params: dict):
    if not db: return []
    engine = db._engine
    with engine.connect() as conn:
        result = conn.execute(text(query), params)
        return result.fetchall()

def make_in_clause(values):
    placeholders = ",".join([f":v{i}" for i in range(len(values))])
    params = {f"v{i}": v for i, v in enumerate(values)}
    return placeholders, params

def fetch_min_sip(scheme_ids):
    placeholders, params = make_in_clause(scheme_ids)
    q = f"SELECT s.scheme_name, m.min_sip FROM minimum_investment m JOIN schemes s ON m.scheme_id = s.scheme_id WHERE m.scheme_id IN ({placeholders})"
    return run_query(q, params)

def fetch_expense_ratio(scheme_ids):
    placeholders, params = make_in_clause(scheme_ids)
    q = f"SELECT s.scheme_name, e.expense_ratio FROM expense_exit_tax e JOIN schemes s ON e.scheme_id = s.scheme_id WHERE e.scheme_id IN ({placeholders})"
    return run_query(q, params)

def fetch_exit_load(scheme_ids):
    placeholders, params = make_in_clause(scheme_ids)
    q = f"SELECT s.scheme_name, e.exit_load FROM expense_exit_tax e JOIN schemes s ON e.scheme_id = s.scheme_id WHERE e.scheme_id IN ({placeholders})"
    return run_query(q, params)

def fetch_nav(scheme_ids):
    placeholders, params = make_in_clause(scheme_ids)
    q = f"SELECT s.scheme_name, n.nav_value_raw, n.nav_as_on FROM nav_summary n JOIN schemes s ON n.scheme_id = s.scheme_id WHERE n.scheme_id IN ({placeholders})"
    return run_query(q, params)

def fetch_fund_managers(scheme_ids):
    placeholders, params = make_in_clause(scheme_ids)
    q = f"SELECT s.scheme_name, f.manager_name FROM fund_managers f JOIN schemes s ON f.scheme_id = s.scheme_id WHERE f.scheme_id IN ({placeholders})"
    return run_query(q, params)

def fetch_returns(scheme_ids):
    placeholders, params = make_in_clause(scheme_ids)
    q = f"""
    SELECT s.scheme_name, r.period_label, r.value
    FROM return_values r
    JOIN schemes s ON r.scheme_id = s.scheme_id
    WHERE r.scheme_id IN ({placeholders})
    AND r.metric_label = 'Fund returns'
    ORDER BY s.scheme_name, r.period_label
    """
    return run_query(q, params)

def fetch_holdings(scheme_ids):
    placeholders, params = make_in_clause(scheme_ids)
    q = f"""
    SELECT s.scheme_name, h.name, h.assets
    FROM holdings h
    JOIN schemes s ON h.scheme_id = s.scheme_id
    WHERE h.scheme_id IN ({placeholders})
    ORDER BY s.scheme_name, CAST(REPLACE(h.assets, '%', '') AS FLOAT) DESC
    LIMIT 10
    """
    return run_query(q, params)

def fetch_ratios(scheme_ids):
    placeholders, params = make_in_clause(scheme_ids)
    q = f"""
    SELECT s.scheme_name, a.alpha, a.beta, a.sharpe_ratio, a.standard_deviation
    FROM advanced_ratios a
    JOIN schemes s ON a.scheme_id = s.scheme_id
    WHERE a.scheme_id IN ({placeholders})
    """
    return run_query(q, params)

In [35]:
# ----------------------------
# RESPONSE FORMATTING
# ----------------------------
OPINIONATED_RESPONSE = (
    "I can’t help with investment advice, recommendations, or performance comparisons. "
    "I’m designed to answer factual questions about Parag Parikh mutual fund schemes only. "
    f"Last updated from sources: {SOURCE_LINK}\n"
    f"Data as of: {LAST_UPDATED_DATE}"
)

def format_answer(rows, intent):
    footer = f"\nLast updated from sources: {SOURCE_LINK}\nData as of: {LAST_UPDATED_DATE}"
    
    if not rows:
        return f"I could not find the requested information in the structured database. " + footer

    if intent == "NAV":
        scheme, nav, date = rows[0]
        return f"The latest NAV for {scheme} is {nav} as of {date}. " + footer

    # General Single-Row Handler
    if len(rows) == 1 and len(rows[0]) == 2:
        scheme, value = rows[0]
        return f"The requested information for {scheme} is {value}. " + footer
        
    # List Handler
    lines = [f"- {r[0]}: {r[1]}" + (f" ({r[2]})" if len(r) > 2 else "") for r in rows]
    return "Here is the requested information:\n" + "\n".join(lines) + footer


In [36]:
# ----------------------------
# MAIN EXECUTION CHAIN
# ----------------------------
def structured_rag_chain(query: str) -> str | None:
    if detect_pii(query):
        return PII_REFUSAL_MESSAGE

    intent = detect_intent(query)
    
    if intent not in STRUCTURED_INTENTS:
        return None

    if intent == "OPINIONATED":
        return OPINIONATED_RESPONSE

    schemes = resolve_schemes(query, intent)
    if not schemes:
        return None

    scheme_ids = [s["scheme_id"] for s in schemes]
    
    if intent == "MIN_SIP": rows = fetch_min_sip(scheme_ids)
    elif intent == "EXPENSE_RATIO": rows = fetch_expense_ratio(scheme_ids)
    elif intent == "EXIT_LOAD": rows = fetch_exit_load(scheme_ids)
    elif intent == "NAV": rows = fetch_nav(scheme_ids)
    elif intent == "FUND_MANAGER": rows = fetch_fund_managers(scheme_ids)
    elif intent == "RETURNS": rows = fetch_returns(scheme_ids)
    elif intent == "HOLDINGS": rows = fetch_holdings(scheme_ids)
    elif intent == "RISK_RATIOS": rows = fetch_ratios(scheme_ids)
    else: return None

    return format_answer(rows, intent)


In [37]:
test_questions = [

    # # 1️⃣ NAV (single-scheme, numeric, date-based)
    # "What is the NAV of Parag Parikh Flexi Cap Fund and Parag Parikh ELSS Fund?",

    # # 2️⃣ Minimum SIP (single-scheme)
    # "What is the minimum SIP amount for Parag Parikh ELSS Fund?",

    # # 3️⃣ Expense Ratio (single-scheme)
    # "Expense ratio of Parag Parikh Liquid Fund",

    # # 4️⃣ Exit Load (single-scheme)
    # "What is the exit load for Parag Parikh Flexi Cap Fund?",

    # # 5️⃣ Fund Manager (single-scheme, multi-row possible)
    # "Who manages Parag Parikh Flexi Cap Fund?",

    # # 6️⃣ Returns (single-scheme, time-period based)
    # "What are the returns of Parag Parikh ELSS Fund?",

    # # 7️⃣ Holdings (single-scheme, multi-row table)
    # "What are the top holdings of Parag Parikh Flexi Cap Fund?",

    # # 8️⃣ Risk Ratios (single-scheme, numeric ratios)
    # "What are the risk ratios of Parag Parikh Flexi Cap Fund?",

    # # 9️⃣ Opinionated (should be refused explicitly)
    # "Which Parag Parikh fund is best?",

    # # 🔟 PII Guard (must trigger PII refusal)
    # "My PAN is ABCDE1234F, what is the NAV of Parag Parikh Flexi Cap Fund?"
]

# for question in test_questions:
#     print("\n\n")
#     print("Question:", question)
#     print("Answer:", structured_rag_chain(question))


### Unstructured RAG Chatbot (LLM + VECTOR DB + EMBEDDINGS)

In [38]:
# ----------------------------
# GLOBAL CONFIGURATION
# ----------------------------
load_dotenv()

# Keys
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    warnings.warn("⚠️ OPENAI_API_KEY not set. LLM features will fail.")

# Paths
DATA_DIR = Path(r"E:\ML Projects\RAG_Chatbot_PP_MF\project_files\data\json_files")
JSON_PATH = DATA_DIR / "chunkable_ppfas.json"

# Models
EMBEDDING_MODEL_NAME = "text-embedding-3-small"
LLM_MODEL_NAME = "gpt-4o-mini"

# Retrieval Settings
RETRIEVAL_K = 5
WEIGHTS = {"bm25": 0.4, "vector": 0.6}

# Constants
AMC_NAME = "PPFAS Mutual Fund"
SOURCE_LINK = "https://groww.in/mutual-funds/amc/ppfas-mutual-funds"

# Chunking Config
SECTIONS_TO_CHUNK = {
    "investment_objective": {"chunk_size": 800, "chunk_overlap": 120}
}
SECTIONS_NO_CHUNK = {"overview", "faq", "contact_details"}
KEYS_TO_REMOVE = {
    "holdings", "advanced_ratios", "returns_and_rankings",
    "expense_exit_tax", "fund_management", "minimum_investment"
}

In [39]:
# ----------------------------
# 1. DATA LOADING & PREP
# ----------------------------
def load_and_prep_data() -> (List[Document], str):
    """
    Loads JSON, chunks it, and returns a list of Documents + last_updated_date.
    """
    if not JSON_PATH.exists():
        raise FileNotFoundError(f"❌ JSON file not found at {JSON_PATH}")

    with open(JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not data:
        raise ValueError("❌ JSON file is empty")

    # Get Date
    last_updated = data[0].get("overview", {}).get("nav", {}).get("as_on", "Unknown Date")
    print(f"✅ Data loaded. Last Updated: {last_updated}")

    documents = []
    
    # Base Document Creation
    for scheme in data:
        scheme_id = scheme["scheme_id"]
        
        # Base metadata helper
        def get_meta(section):
            return {
                "scheme_id": scheme_id,
                "section": section,
                "amc_name": AMC_NAME,
                "source": SOURCE_LINK,
                "last_updated": last_updated
            }

        # Overview
        if "overview" in scheme:
            documents.append(Document(
                page_content=json.dumps(scheme["overview"], indent=2),
                metadata=get_meta("overview")
            ))

        # Investment Objective
        if "fund_house_and_investment_objective" in scheme:
             documents.append(Document(
                page_content=scheme["fund_house_and_investment_objective"],
                metadata=get_meta("investment_objective")
            ))

        # FAQs
        for faq in scheme.get("faqs", {}).get("items", []):
            text = f"Question: {faq.get('question')}\n\nAnswer: {faq.get('answer')}"
            documents.append(Document(
                page_content=text,
                metadata=get_meta("faq")
            ))

    # Chunking Logic (Reuse from notebook)
    final_docs = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=SECTIONS_TO_CHUNK["investment_objective"]["chunk_size"],
        chunk_overlap=SECTIONS_TO_CHUNK["investment_objective"]["chunk_overlap"]
    )

    for doc in documents:
        section = doc.metadata["section"]
        
        if section in SECTIONS_TO_CHUNK:
            chunks = splitter.split_text(doc.page_content)
            for i, chunk in enumerate(chunks):
                meta = doc.metadata.copy()
                meta["chunk_index"] = i
                final_docs.append(Document(page_content=chunk, metadata=meta))
        else:
            final_docs.append(doc)

    return final_docs, last_updated

In [40]:
# ----------------------------
# 2. RETRIEVAL SYSTEM (HYBRID)
# ----------------------------
# Initialize resources globally to avoid reloading on every call
print("⏳ Initializing Retriever...")
try:
    _DOCS, _LAST_UPDATED = load_and_prep_data()

    # BM25
    _BM25_CORPUS = [doc.page_content.lower().split() for doc in _DOCS]
    _BM25_INDEX = BM25Okapi(_BM25_CORPUS)

    # Vector Store (FAISS)
    _EMBEDDINGS = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME)
    _VECTOR_STORE = FAISS.from_documents(_DOCS, _EMBEDDINGS)
    
    print("✅ Retriever Initialized (BM25 + FAISS)")
except Exception as e:
    print(f"❌ Failed to initialize retriever: {e}")
    _DOCS, _LAST_UPDATED, _BM25_INDEX, _VECTOR_STORE = [], "Error", None, None


def split_text_tokenized(text):
    return text.lower().split()

def hybrid_retrieve(query: str, k: int = RETRIEVAL_K) -> List[Document]:
    if not _BM25_INDEX or not _VECTOR_STORE:
        return []

    # 1. BM25 Search
    bm25_scores = _BM25_INDEX.get_scores(split_text_tokenized(query))
    # Pair (index, score) and sort
    bm25_ranked = sorted(enumerate(bm25_scores), key=lambda x: x[1], reverse=True)[:k*2]
    
    # Normalize BM25 (simple min-max of top results)
    bm25_results = {}
    if bm25_ranked:
        max_score = bm25_ranked[0][1] or 1.0
        for idx, score in bm25_ranked:
            doc = _DOCS[idx]
            bm25_results[id(doc)] = {
                "doc": doc,
                "score": (score / max_score) * WEIGHTS["bm25"]
            }

    # 2. Vector Search
    vector_results_raw = _VECTOR_STORE.similarity_search_with_score(query, k=k*2)
    # FAISS returns L2 distance (lower is better). We need similarity.
    # Convert distance to similarity score approx: 1 / (1 + dist) or specific range inverted.
    # For now, let's normalize simply by max distance in batch.
    
    combined_results = bm25_results.copy()
    
    if vector_results_raw:
        max_dist = max(score for _, score in vector_results_raw) or 1.0
        for doc, dist in vector_results_raw:
            # Simple inversion: (1 - dist/max) * weight
            # Note: This is a rough heuristic.
            sim_score = (1 - (dist / (max_dist + 0.01))) * WEIGHTS["vector"]
            
            doc_id = id(doc)
            if doc_id in combined_results:
                combined_results[doc_id]["score"] += sim_score
            else:
                combined_results[doc_id] = {
                    "doc": doc,
                    "score": sim_score
                }

    # Sort final
    final_ranked = sorted(combined_results.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in final_ranked[:k]]


⏳ Initializing Retriever...
✅ Data loaded. Last Updated: 02 Jan 2026
✅ Retriever Initialized (BM25 + FAISS)


In [41]:
# ----------------------------
# 3. UNSTRUCTURED CHAIN
# ----------------------------
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a factual financial assistant for Parag Parikh Mutual Fund schemes.

Rules:
- Answer ONLY using the provided context.
- If the answer is not present, say "I could not find the info in the documents."
- Do NOT provide investment advice or comparisons.
- Be concise and factual.

Context:
{context}

User Question:
{question}

Answer:
""")

_LLM = ChatOpenAI(model=LLM_MODEL_NAME, temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def unstructured_rag_chain(query: str) -> str:
    # Retrieve
    docs = hybrid_retrieve(query)
    if not docs:
        return f"I could not find relevant information.\nLast updated from sources: {SOURCE_LINK}\nData as of: {_LAST_UPDATED}"

    # Generate
    chain = (
        {"context": lambda x: format_docs(docs), "question": RunnablePassthrough()}
        | RAG_PROMPT
        | _LLM
    )
    
    response = chain.invoke(query)
    content = response.content if hasattr(response, 'content') else str(response)

    return (
        content.strip()
        + f"\n\nLast updated from sources: {SOURCE_LINK}"
        + f"\nData as of: {_LAST_UPDATED}"
    )

In [42]:
# ----------------------------
# 4. FINAL COMBINED CHAIN
# ----------------------------
def final_rag_chain(query: str) -> str:
    # 1. Try Structured
    try:
        struct_res = structured_rag_chain(query)
        if struct_res:
            return struct_res
    except Exception as e:
        print(f"⚠️ Structured RAG Error: {e}")

    # 2. Fallback to Unstructured
    return unstructured_rag_chain(query)

In [ ]:
    print("-" * 50)
    print("🧪 TESTING FINAL RAG CHAIN")
    print("-" * 50)
    
    test_qs = [
        "How to invest in PPFAS Flexi Cap fund?",        # Unstructured
    ]
    
    for q in test_qs:
        print(f"\n❓ Q: {q}")
        print(f"🤖 A: {final_rag_chain(q)}")


--------------------------------------------------
🧪 TESTING FINAL RAG CHAIN
--------------------------------------------------

❓ Q: How to invest in PPFAS Flexi Cap fund?
🤖 A: I could not find the info in the documents.

Last updated from sources: https://groww.in/mutual-funds/amc/ppfas-mutual-funds
Data as of: 02 Jan 2026
